In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
clip_df = pd.read_csv('../clips_metadata.csv',sep=",")

N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 41

print(f'Total clips: {len(clip_df)}')              # ← clip_df not df
print(f'Classes: {clip_df["species"].nunique()}')   # ← species not emotion

Total clips: 49682
Classes: 41


In [4]:
le = LabelEncoder()
clip_df['label'] = le.fit_transform(clip_df['species'])

In [5]:
class AviaNetDataset(Dataset):
    def __init__(self, clip_df, augment_data=False):
        self.clip_df = clip_df.reset_index(drop=True)
        self.augment_data = augment_data
    def __len__(self):
        return len(self.clip_df)
    def __getitem__(self, idx):
        row = self.clip_df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)

        return tensor, label

    def augment(self, y, sr):
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
        shift = np.random.randint(0, max(1, sr // 4))
        y = np.roll(y, shift)
        # removed pitch_shift — too slow at this scale
        return y



print('Dataset class defined')


Dataset class defined


In [6]:
train_df, test_df = train_test_split(
    clip_df, test_size=0.2, random_state=42, stratify=clip_df['label']
)

train_df_sample = train_df.sample(n=10000, random_state=42)

train_dataset = AviaNetDataset(train_df_sample, augment_data=True)
test_dataset = AviaNetDataset(test_df, augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 10000
Test samples:  9937
Train batches: 157
Test batches:  156
Batch X shape: torch.Size([64, 1, 128, 216])
Batch y shape: torch.Size([64])


In [7]:
class AviaNet(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super(AviaNet,self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 =  nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(55296, 512),    # ← fix from 128*16*16
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x

In [8]:
model = AviaNet(num_classes=NUM_CLASSES).to(device)

with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 216).to(device)
    out = model.conv1(dummy)
    print(f'After conv1: {out.shape}')
    out = model.conv2(out)
    print(f'After conv2: {out.shape}')
    out = model.conv3(out)
    print(f'After conv3: {out.shape}')
    print(f'Flattened: {out.view(1, -1).shape[1]}')

After conv1: torch.Size([1, 32, 64, 108])
After conv2: torch.Size([1, 64, 32, 54])
After conv3: torch.Size([1, 128, 16, 27])
Flattened: 55296


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 5.2649 | Test Acc: 0.1666
Epoch [2/30] Loss: 3.1862 | Test Acc: 0.1514
Epoch [3/30] Loss: 3.1258 | Test Acc: 0.1355
Epoch [4/30] Loss: 3.0849 | Test Acc: 0.1350
Epoch [5/30] Loss: 3.0517 | Test Acc: 0.1278
Epoch [6/30] Loss: 3.0106 | Test Acc: 0.1210
Epoch [7/30] Loss: 2.9955 | Test Acc: 0.1367
Epoch [8/30] Loss: 2.9720 | Test Acc: 0.1484
Epoch [9/30] Loss: 2.9531 | Test Acc: 0.1560
Epoch [10/30] Loss: 2.9316 | Test Acc: 0.1486
Epoch [11/30] Loss: 2.8688 | Test Acc: 0.1449
Epoch [12/30] Loss: 2.8360 | Test Acc: 0.1460
Epoch [13/30] Loss: 2.8261 | Test Acc: 0.1282
Epoch [14/30] Loss: 2.8251 | Test Acc: 0.1385
Epoch [15/30] Loss: 2.8154 | Test Acc: 0.1325
Epoch [16/30] Loss: 2.7933 | Test Acc: 0.1406
Epoch [17/30] Loss: 2.7727 | Test Acc: 0.1442
Epoch [18/30] Loss: 2.7478 | Test Acc: 0.1483
Epoch [19/30] Loss: 2.7337 | Test Acc: 0.1460
Epoch [20/30] Loss: 2.7225 | Test Acc: 0.1448
Epoch [21/30] Loss: 2.6809 | Test Acc: 0.1603
Epoch [22/30] Loss: 2.6793 | Test Acc: 0.15